<a href="https://colab.research.google.com/github/sreevarshini22/CODSOFT/blob/main/Task3/Churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)
from sklearn.pipeline import Pipeline

# SECTION 1 - SYNTHETIC DATASET GENERATION

def build_customer_dataset(num_records=5000, seed=42):
    """
    Creates a realistic synthetic dataset simulating customers of a
    subscription-based service such as a SaaS platform or streaming app.

    Features include:
        - Customer demographics (age, gender, location)
        - Subscription details (contract type, tenure, billing)
        - Behavioral signals (login frequency, support calls, usage)

    The churn label is generated using a probability function that
    weighs these features according to real-world churn drivers.

    Parameters
    ----------
    num_records : int
        Number of customer rows to generate.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame
        Full dataset with features and binary churn label.
    """
    rand = np.random.default_rng(seed)

    # --- Demographics ---
    customer_age      = rand.integers(18, 70, num_records)
    customer_gender   = rand.choice(
        ["Male", "Female", "Other"], num_records, p=[0.48, 0.48, 0.04]
    )
    customer_region   = rand.choice(
        ["Urban", "Suburban", "Rural"], num_records, p=[0.50, 0.35, 0.15]
    )

    # --- Contract & Billing ---
    subscription_type = rand.choice(
        ["Month-to-Month", "One Year", "Two Year"],
        num_records, p=[0.55, 0.30, 0.15]
    )
    months_active     = rand.integers(1, 73, num_records)
    bill_per_month    = rand.uniform(20, 120, num_records).round(2)
    bill_total        = (
        bill_per_month * months_active * rand.uniform(0.95, 1.05, num_records)
    ).round(2)
    active_products   = rand.integers(1, 6, num_records)
    payment_channel   = rand.choice(
        ["Credit Card", "Bank Transfer", "Electronic Check", "Mailed Check"],
        num_records, p=[0.30, 0.25, 0.30, 0.15]
    )
    digital_billing   = rand.choice([0, 1], num_records, p=[0.35, 0.65])

    # Behavioral Usage
    data_used_gb      = rand.uniform(1, 300, num_records).round(1)
    logins_per_month  = rand.integers(0, 31, num_records)
    support_contacts  = rand.integers(0, 11, num_records)
    missed_payments   = rand.integers(0, 6, num_records)
    engagement_score  = rand.uniform(0, 10, num_records).round(2)

    #  Churn Probability Formula
    # Customers on month-to-month contracts, with high support calls,
    # missed payments, or low logins are more likely to churn.
    raw_churn_prob = (
        0.05
        + 0.25 * (subscription_type == "Month-to-Month")
        - 0.10 * (subscription_type == "Two Year")
        + 0.03 * support_contacts
        + 0.02 * missed_payments
        - 0.005 * logins_per_month
        - 0.002 * months_active
        + 0.01  * (customer_region == "Rural")
        - 0.002 * engagement_score
    )
    raw_churn_prob = np.clip(raw_churn_prob, 0.02, 0.85)
    churn_label    = rand.binomial(1, raw_churn_prob).astype(int)

    dataset = pd.DataFrame({
        "Age":              customer_age,
        "Gender":           customer_gender,
        "Region":           customer_region,
        "ContractType":     subscription_type,
        "TenureMonths":     months_active,
        "MonthlyCharges":   bill_per_month,
        "TotalCharges":     bill_total,
        "NumProducts":      active_products,
        "PaymentMethod":    payment_channel,
        "PaperlessBilling": digital_billing,
        "MonthlyUsageGB":   data_used_gb,
        "LoginFrequency":   logins_per_month,
        "SupportCalls":     support_contacts,
        "LatePayments":     missed_payments,
        "EngagementScore":  engagement_score,
        "Churn":            churn_label,
    })

    return dataset


# SECTION 2 - PREPROCESSING

def prepare_data(raw_df):
    """
    Encodes categorical columns, splits features and labels,
    and returns stratified train/test sets.

    Parameters
    ----------
    raw_df : pd.DataFrame
        The full raw dataset including the Churn column.

    Returns
    -------
    tuple : X_train, X_test, y_train, y_test, list of feature names
    """
    working_df   = raw_df.copy()
    label_enc    = LabelEncoder()
    cat_columns  = ["Gender", "Region", "ContractType", "PaymentMethod"]

    for col in cat_columns:
        working_df[col] = label_enc.fit_transform(working_df[col])

    feature_matrix = working_df.drop("Churn", axis=1)
    target_vector  = working_df["Churn"]
    col_names      = feature_matrix.columns.tolist()

    X_train, X_test, y_train, y_test = train_test_split(
        feature_matrix,
        target_vector,
        test_size=0.20,
        random_state=42,
        stratify=target_vector
    )

    return X_train, X_test, y_train, y_test, col_names


# SECTION 3 - MODEL DEFINITIONS


def define_classifiers():
    """
    Defines three classifiers with tuned hyperparameters.

    Logistic Regression uses a StandardScaler pipeline since it is
    sensitive to feature scale. Tree-based models (RF, GBM) do not
    require scaling.

    Returns
    -------
    dict : { model_name: fitted_or_pipeline_object }
    """
    lr_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            C=0.5,
            solver="lbfgs",
            random_state=42
        ))
    ])

    rf_classifier = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    gb_classifier = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42
    )

    return {
        "Logistic Regression": lr_pipeline,
        "Random Forest":       rf_classifier,
        "Gradient Boosting":   gb_classifier,
    }


# SECTION 4 - EVALUATION

def score_classifier(trained_model, X_test, y_test, label="Model"):
    """
    Evaluates a fitted classifier on the test set.
    Prints a full classification report and returns metric scores.

    Parameters
    ----------
    trained_model : fitted sklearn estimator
    X_test        : pd.DataFrame
    y_test        : pd.Series
    label         : str, model display name

    Returns
    -------
    tuple : (metrics_dict, predicted_labels, predicted_probabilities)
    """
    pred_labels = trained_model.predict(X_test)
    pred_probs  = trained_model.predict_proba(X_test)[:, 1]

    score_dict = {
        "Accuracy":  accuracy_score(y_test, pred_labels),
        "Precision": precision_score(y_test, pred_labels, zero_division=0),
        "Recall":    recall_score(y_test, pred_labels, zero_division=0),
        "F1 Score":  f1_score(y_test, pred_labels, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_test, pred_probs),
    }

    print("\n" + "-" * 52)
    print("  Results: " + label)
    print("-" * 52)
    for metric_name, metric_val in score_dict.items():
        print(f"  {metric_name:<12}: {metric_val:.4f}")
    print()
    print(classification_report(
        y_test, pred_labels,
        target_names=["Retained", "Churned"]
    ))

    return score_dict, pred_labels, pred_probs


# SECTION 5 - VISUALIZATION

BG_COLOR    = "#0A0F1E"
CARD_COLOR  = "#12182F"
TEXT_COLOR  = "#E8EEFF"
GRID_COLOR  = "#1E2A50"
ACCENT_BLUE = "#4A6CF7"
ACCENT_ORG  = "#F7A64A"
COLOR_SCALE = ["#0A0F1E", "#1A1F3C", "#2E3A6E", "#4A6CF7", "#7B96FF", "#A8C0FF"]
MODEL_CLRS  = [ACCENT_BLUE, ACCENT_ORG, "#4AFCC7"]


def apply_axis_style(ax, title="", xlabel="", ylabel=""):
    """Applies consistent dark-theme styling to a matplotlib axis."""
    ax.set_facecolor(CARD_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COLOR)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    if title:
        ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=8)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=8)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.5, alpha=0.7)


def render_dashboard(
    all_results,
    trained_clf_map,
    X_tr, X_te, y_tr, y_te,
    col_names,
    source_df,
    save_path="churn_dashboard.png"
):
    """
    Builds and saves a comprehensive analytics dashboard.
    """
    plt.rcParams.update({
        "font.family":       "DejaVu Sans",
        "text.color":        TEXT_COLOR,
        "axes.facecolor":    CARD_COLOR,
        "figure.facecolor":  BG_COLOR,
        "savefig.facecolor": BG_COLOR,
    })

    fig = plt.figure(figsize=(22, 26))
    fig.patch.set_facecolor(BG_COLOR)

    fig.text(
        0.5, 0.985,
        "CUSTOMER CHURN PREDICTION DASHBOARD",
        ha="center", va="top",
        fontsize=21, fontweight="bold", color=TEXT_COLOR
    )
    fig.text(
        0.5, 0.974,
        "Logistic Regression  |  Random Forest  |  Gradient Boosting",
        ha="center", va="top",
        fontsize=11, color=COLOR_SCALE[4]
    )

    layout = gridspec.GridSpec(
        5, 4, figure=fig,
        hspace=0.52, wspace=0.38,
        left=0.06, right=0.97,
        top=0.96, bottom=0.03
    )

    clf_names   = list(all_results.keys())
    metric_keys = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

    # Row 0 - Class distribution | Metric bars | ROC-AUC summary
    ax_dist      = fig.add_subplot(layout[0, 0])
    class_counts = source_df["Churn"].value_counts()
    dist_bars    = ax_dist.bar(
        ["Retained", "Churned"],
        class_counts.values,
        color=[COLOR_SCALE[3], "#F7664A"],
        edgecolor=BG_COLOR,
        linewidth=1.5,
        width=0.5
    )
    for bar_obj, cnt in zip(dist_bars, class_counts.values):
        ax_dist.text(
            bar_obj.get_x() + bar_obj.get_width() / 2,
            cnt + 30, str(cnt),
            ha="center", color=TEXT_COLOR,
            fontsize=9, fontweight="bold"
        )
    apply_axis_style(ax_dist, "Class Distribution", "", "Count")
    ax_dist.grid(False)

    ax_compare = fig.add_subplot(layout[0, 1:3])
    x_pos  = np.arange(len(metric_keys))
    bar_w  = 0.22
    for idx, (clf_name, (mets, _, _)) in enumerate(all_results.items()):
        vals = [mets[k] for k in metric_keys]
        ax_compare.bar(
            x_pos + idx * bar_w, vals, width=bar_w,
            label=clf_name, color=MODEL_CLRS[idx],
            edgecolor=BG_COLOR, linewidth=0.8, alpha=0.92
        )
    ax_compare.set_xticks(x_pos + bar_w)
    ax_compare.set_xticklabels(metric_keys, fontsize=8, color=TEXT_COLOR)
    ax_compare.set_ylim(0, 1.12)
    ax_compare.legend(fontsize=8, facecolor=CARD_COLOR,
                      edgecolor=GRID_COLOR, labelcolor=TEXT_COLOR)
    apply_axis_style(ax_compare, "Model Performance Comparison", "", "Score")

    ax_auc   = fig.add_subplot(layout[0, 3])
    auc_vals = [all_results[n][0]["ROC-AUC"] for n in clf_names]
    hbars    = ax_auc.barh(clf_names, auc_vals, color=MODEL_CLRS,
                           edgecolor=BG_COLOR, linewidth=1)
    for hbar, v in zip(hbars, auc_vals):
        ax_auc.text(
            v - 0.04,
            hbar.get_y() + hbar.get_height() / 2,
            f"{v:.3f}",
            va="center", color="white",
            fontsize=9, fontweight="bold"
        )
    ax_auc.set_xlim(0, 1)
    apply_axis_style(ax_auc, "ROC-AUC Summary", "AUC", "")
    ax_auc.grid(axis="x", color=GRID_COLOR, linewidth=0.5, alpha=0.7)
    ax_auc.grid(False, axis="y")

    # Row 1 - ROC Curves | Precision-Recall Curves
    ax_roc = fig.add_subplot(layout[1, 0:2])
    for idx, (clf_name, (mets, _, proba)) in enumerate(all_results.items()):
        fpr, tpr, _ = roc_curve(y_te, proba)
        ax_roc.plot(fpr, tpr, color=MODEL_CLRS[idx], lw=2,
                    label=f"{clf_name}  (AUC={mets['ROC-AUC']:.3f})")
    ax_roc.plot([0, 1], [0, 1], "--", color=GRID_COLOR, lw=1)
    ax_roc.legend(fontsize=8, facecolor=CARD_COLOR,
                  edgecolor=GRID_COLOR, labelcolor=TEXT_COLOR)
    apply_axis_style(ax_roc, "ROC Curves", "False Positive Rate", "True Positive Rate")

    ax_pr = fig.add_subplot(layout[1, 2:4])
    for idx, (clf_name, (_, _, proba)) in enumerate(all_results.items()):
        prec_vals, rec_vals, _ = precision_recall_curve(y_te, proba)
        ax_pr.plot(rec_vals, prec_vals, color=MODEL_CLRS[idx], lw=2, label=clf_name)
    ax_pr.legend(fontsize=8, facecolor=CARD_COLOR,
                 edgecolor=GRID_COLOR, labelcolor=TEXT_COLOR)
    apply_axis_style(ax_pr, "Precision-Recall Curves", "Recall", "Precision")

    # Row 2 - Confusion Matrices | Cross-Validation
    for idx, (clf_name, (_, preds, _)) in enumerate(all_results.items()):
        ax_cm     = fig.add_subplot(layout[2, idx])
        cm_matrix = confusion_matrix(y_te, preds)
        sns.heatmap(
            cm_matrix, annot=True, fmt="d", cmap="Blues",
            ax=ax_cm, cbar=False, linewidths=0.5,
            annot_kws={"size": 11, "color": "white"},
            xticklabels=["Retained", "Churned"],
            yticklabels=["Retained", "Churned"]
        )
        ax_cm.set_facecolor(CARD_COLOR)
        ax_cm.tick_params(colors=TEXT_COLOR, labelsize=7.5)
        for spine in ax_cm.spines.values():
            spine.set_edgecolor(GRID_COLOR)
        ax_cm.set_title(
            "Confusion Matrix\n" + clf_name,
            fontsize=9, fontweight="bold", color=TEXT_COLOR
        )
        ax_cm.set_xlabel("Predicted", color=TEXT_COLOR, fontsize=8)
        ax_cm.set_ylabel("Actual",    color=TEXT_COLOR, fontsize=8)

    ax_cv       = fig.add_subplot(layout[2, 3])
    cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for idx, clf_name in enumerate(clf_names):
        clf_obj   = trained_clf_map[clf_name]
        cv_scores = cross_val_score(
            clf_obj, X_tr, y_tr,
            cv=cv_strategy, scoring="roc_auc", n_jobs=-1
        )
        ax_cv.errorbar(
            idx, cv_scores.mean(), yerr=cv_scores.std(),
            fmt="o", color=MODEL_CLRS[idx], capsize=5, markersize=8,
            label=(clf_name.split()[0]
                   + "\n" + f"{cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
        )
    ax_cv.set_xticks(range(len(clf_names)))
    ax_cv.set_xticklabels(
        [n.split()[0] for n in clf_names],
        fontsize=8, color=TEXT_COLOR
    )
    ax_cv.set_ylim(0.6, 1.0)
    apply_axis_style(ax_cv, "5-Fold Cross-Validation (ROC-AUC)", "", "AUC")

    # Row 3 - Feature Importance
    ax_rf_imp = fig.add_subplot(layout[3, 0:2])
    rf_model  = trained_clf_map["Random Forest"]
    rf_imp    = pd.Series(rf_model.feature_importances_, index=col_names)
    rf_imp.nlargest(12).sort_values().plot(
        kind="barh", ax=ax_rf_imp,
        color=ACCENT_ORG, edgecolor=BG_COLOR, linewidth=0.8
    )
    apply_axis_style(ax_rf_imp,
                     "Random Forest - Feature Importance (Top 12)",
                     "Importance Score", "")
    ax_rf_imp.tick_params(axis="y", labelsize=7.5)
    ax_rf_imp.grid(axis="x", color=GRID_COLOR, linewidth=0.5, alpha=0.7)
    ax_rf_imp.grid(False, axis="y")

    ax_gb_imp = fig.add_subplot(layout[3, 2:4])
    gb_model  = trained_clf_map["Gradient Boosting"]
    gb_imp    = pd.Series(gb_model.feature_importances_, index=col_names)
    gb_imp.nlargest(12).sort_values().plot(
        kind="barh", ax=ax_gb_imp,
        color="#4AFCC7", edgecolor=BG_COLOR, linewidth=0.8
    )
    apply_axis_style(ax_gb_imp,
                     "Gradient Boosting - Feature Importance (Top 12)",
                     "Importance Score", "")
    ax_gb_imp.tick_params(axis="y", labelsize=7.5)
    ax_gb_imp.grid(axis="x", color=GRID_COLOR, linewidth=0.5, alpha=0.7)
    ax_gb_imp.grid(False, axis="y")

    # Row 4 - Probability Distribution | Score Card Table
    ax_prob       = fig.add_subplot(layout[4, 0:2])
    gb_proba_vals = all_results["Gradient Boosting"][2]
    ax_prob.hist(gb_proba_vals[y_te == 0], bins=40, alpha=0.7,
                 color=COLOR_SCALE[3], label="Retained", edgecolor=BG_COLOR)
    ax_prob.hist(gb_proba_vals[y_te == 1], bins=40, alpha=0.7,
                 color="#F7664A", label="Churned", edgecolor=BG_COLOR)
    ax_prob.axvline(0.5, color=TEXT_COLOR, lw=1.5,
                    linestyle="--", label="Decision Threshold = 0.5")
    ax_prob.legend(fontsize=8, facecolor=CARD_COLOR,
                   edgecolor=GRID_COLOR, labelcolor=TEXT_COLOR)
    apply_axis_style(
        ax_prob,
        "Gradient Boosting - Predicted Churn Probability Distribution",
        "Predicted Probability", "Count"
    )

    ax_table = fig.add_subplot(layout[4, 2:4])
    ax_table.set_facecolor(CARD_COLOR)
    ax_table.axis("off")

    header_row = ["Metric"] + clf_names
    table_rows = []
    for mk in metric_keys:
        row = [mk] + [f"{all_results[n][0][mk]:.4f}" for n in clf_names]
        table_rows.append(row)

    score_table = ax_table.table(
        cellText=table_rows,
        colLabels=header_row,
        cellLoc="center",
        loc="center",
        bbox=[0, 0, 1, 1]
    )
    score_table.auto_set_font_size(False)
    score_table.set_fontsize(9)

    for (row_idx, col_idx), cell in score_table.get_celld().items():
        cell.set_facecolor(CARD_COLOR if row_idx % 2 == 0 else "#161D38")
        cell.set_edgecolor(GRID_COLOR)
        if row_idx == 0:
            cell.set_text_props(color=ACCENT_ORG, fontweight="bold")
        else:
            cell.set_text_props(color=TEXT_COLOR)

    ax_table.set_title(
        "Score Card - All Models",
        fontsize=10, fontweight="bold",
        color=TEXT_COLOR, pad=8
    )

    # Ensure the output directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    print("\nDashboard saved to: " + save_path)
    plt.close()


# SECTION 6 - MAIN EXECUTION


if __name__ == "__main__":

    DASHBOARD_OUT = "/mnt/user-data/outputs/churn_dashboard.png"
    MODEL_OUT     = "/mnt/user-data/outputs/best_churn_model.pkl"

    print("=" * 58)
    print("   CUSTOMER CHURN PREDICTION PIPELINE")
    print("=" * 58)

    # Step 1 - Generate dataset
    print("\n[Step 1/4]  Generating customer dataset...")
    customer_df = build_customer_dataset(num_records=5000, seed=42)
    churn_rate  = customer_df["Churn"].mean()
    print(f"            Records   : {len(customer_df)}")
    print(f"            Features  : {customer_df.shape[1] - 1}")
    print(f"            Churn Rate: {churn_rate:.2%}")

    # Step 2 - Preprocess
    print("\n[Step 2/4]  Preprocessing data...")
    X_train, X_test, y_train, y_test, feature_cols = prepare_data(customer_df)
    print(f"            Train set : {X_train.shape[0]} records")
    print(f"            Test set  : {X_test.shape[0]} records")

    # Step 3 - Train
    print("\n[Step 3/4]  Training classifiers...")
    classifier_map = define_classifiers()
    fitted_clfs    = {}
    evaluation_log = {}

    for clf_name, clf_obj in classifier_map.items():
        print(f"\n            Fitting: {clf_name}")
        clf_obj.fit(X_train, y_train)
        fitted_clfs[clf_name] = clf_obj
        scores, preds, probas = score_classifier(
            clf_obj, X_test, y_test, label=clf_name
        )
        evaluation_log[clf_name] = (scores, preds, probas)

    # Step 4 - Visualize
    print("\n[Step 4/4]  Building analytics dashboard...")
    render_dashboard(
        all_results=evaluation_log,
        trained_clf_map=fitted_clfs,
        X_tr=X_train, X_te=X_test,
        y_tr=y_train, y_te=y_test,
        col_names=feature_cols,
        source_df=customer_df,
        save_path=DASHBOARD_OUT
    )

    # Step 5 - Save best model
    best_clf_name = max(
        evaluation_log,
        key=lambda n: evaluation_log[n][0]["ROC-AUC"]
    )
    best_auc = evaluation_log[best_clf_name][0]["ROC-AUC"]

    print(f"\nBest performing model : {best_clf_name}")
    print(f"ROC-AUC on test set   : {best_auc:.4f}")

    joblib.dump(fitted_clfs[best_clf_name], MODEL_OUT)
    print(f"Model saved to        : {MODEL_OUT}")

    print("\n" + "=" * 58)
    print("   Pipeline complete.")
    print("=" * 58)

   CUSTOMER CHURN PREDICTION PIPELINE

[Step 1/4]  Generating customer dataset...
            Records   : 5000
            Features  : 15
            Churn Rate: 23.40%

[Step 2/4]  Preprocessing data...
            Train set : 4000 records
            Test set  : 1000 records

[Step 3/4]  Training classifiers...

            Fitting: Logistic Regression

----------------------------------------------------
  Results: Logistic Regression
----------------------------------------------------
  Accuracy    : 0.7560
  Precision   : 0.4468
  Recall      : 0.1795
  F1 Score    : 0.2561
  ROC-AUC     : 0.7416

              precision    recall  f1-score   support

    Retained       0.79      0.93      0.85       766
     Churned       0.45      0.18      0.26       234

    accuracy                           0.76      1000
   macro avg       0.62      0.56      0.56      1000
weighted avg       0.71      0.76      0.71      1000


            Fitting: Random Forest

-------------------------